# ДЗ 4. California Housing: EDA и линейная регрессия
**Выполнил:** Хритинин Д. А.

**Задание:**
1. Взять датасет California Housing.
2. Сделать EDA (по аналогии с Титаником).
3. Предсказать цену дома с помощью `LinearRegression` из scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

sns.set_theme(style="whitegrid")

## 1. Загрузка данных

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(housing.DESCR[:1500])

**Описание признаков** (одна строка = один квартал (block group) Калифорнии по переписи 1990 г.):
- `MedInc` — медианный доход в квартале (в десятках тысяч $)
- `HouseAge` — медианный возраст домов
- `AveRooms` — среднее число комнат на домохозяйство
- `AveBedrms` — среднее число спален на домохозяйство
- `Population` — население квартала
- `AveOccup` — среднее число жильцов на домохозяйство
- `Latitude`, `Longitude` — координаты
- **`MedHouseVal`** — целевая переменная: медианная стоимость дома (в сотнях тысяч $)

In [ ]:
df.head()

## 2. EDA
### 2.1. Общая информация, типы и пропуски

In [ ]:
print("Размер:", df.shape)
df.info()

In [ ]:
print("Пропуски по столбцам:")
print(df.isnull().sum())
print("\nДубликатов строк:", df.duplicated().sum())

### 2.2. Описательная статистика

In [ ]:
df.describe().T.round(2)

### 2.3. Распределение целевой переменной

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["MedHouseVal"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Распределение MedHouseVal")
sns.boxplot(x=df["MedHouseVal"], ax=axes[1])
axes[1].set_title("Boxplot MedHouseVal")
plt.show()

print("Домов со стоимостью ровно 5.0 (верхняя граница):", (df["MedHouseVal"] >= 5.0).sum())

Распределение скошено вправо, а в районе 5.0 виден резкий пик: значения цены обрезаны сверху (все дома дороже 500 000 $ записаны как 5.0). Для линейной модели это мешает: реальную цену таких домов она «не видит».

### 2.4. Распределения всех признаков

In [ ]:
df.hist(bins=50, figsize=(15, 10))
plt.suptitle("Гистограммы признаков", y=1.02)
plt.tight_layout(); plt.show()

### 2.5. Выбросы

In [ ]:
features = housing.feature_names
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col, ax in zip(features, axes.flat):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

# Число выбросов по правилу 1.5 IQR
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

df[features].apply(iqr_outliers).rename("выбросов (1.5 IQR)")

In [ ]:
# Самые экстремальные значения
df[["AveRooms", "AveBedrms", "AveOccup", "Population"]].sort_values("AveOccup", ascending=False).head()

В `AveRooms`, `AveBedrms`, `AveOccup` и `Population` есть экстремальные значения (например, сотни жильцов на домохозяйство) — скорее всего, это общежития, гостиницы или ошибки данных.

### 2.6. Корреляции

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Матрица корреляций"); plt.show()

df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)

### 2.7. Зависимость цены от ключевых признаков

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.scatterplot(data=df, x="MedInc", y="MedHouseVal", alpha=0.2, s=8, ax=axes[0])
axes[0].set_title("Цена vs доход")
sns.scatterplot(data=df, x="HouseAge", y="MedHouseVal", alpha=0.2, s=8, ax=axes[1])
axes[1].set_title("Цена vs возраст дома")
sns.scatterplot(data=df[df["AveRooms"] < 15], x="AveRooms", y="MedHouseVal", alpha=0.2, s=8, ax=axes[2])
axes[2].set_title("Цена vs комнаты (AveRooms < 15)")
plt.tight_layout(); plt.show()

In [ ]:
# Средняя цена по группам дохода (аналог группировок в Титанике)
df["IncomeGroup"] = pd.qcut(df["MedInc"], 5, labels=["очень низкий", "низкий", "средний", "высокий", "очень высокий"])
print(df.groupby("IncomeGroup", observed=True)["MedHouseVal"].agg(["mean", "median", "count"]).round(2))

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="IncomeGroup", y="MedHouseVal")
plt.title("Цена дома по группам дохода"); plt.xlabel(""); plt.show()
df = df.drop(columns="IncomeGroup")

### 2.8. География цен

In [ ]:
plt.figure(figsize=(9, 7))
sc = plt.scatter(df["Longitude"], df["Latitude"], c=df["MedHouseVal"], cmap="viridis",
                 s=df["Population"] / 300, alpha=0.4)
plt.colorbar(sc, label="MedHouseVal")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("Стоимость домов на карте Калифорнии (размер точки — население)")
plt.show()

### Выводы по EDA
- 20 640 строк, 8 числовых признаков, **пропусков и категориальных признаков нет** — кодирование и заполнение не нужно.
- Целевая переменная скошена вправо и **обрезана на 5.0**.
- Сильнее всего с ценой связан **медианный доход** (`MedInc`, корреляция ≈ 0.69). Остальные признаки по отдельности связаны с ценой слабо.
- `AveRooms` и `AveBedrms` сильно коррелируют между собой (мультиколлинеарность), `Latitude` и `Longitude` — тоже.
- В `AveRooms`, `AveBedrms`, `AveOccup`, `Population` есть сильные выбросы.
- На карте дорогие дома сосредоточены на побережье (районы Сан-Франциско и Лос-Анджелеса): местоположение важно, но связь с координатами нелинейная.

## 3. Линейная регрессия

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[features]
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
# Масштабирование не меняет предсказания линейной регрессии,
# но делает коэффициенты сравнимыми между собой
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    pred_tr, pred_te = model.predict(X_tr), model.predict(X_te)
    return {
        "Модель": name,
        "MAE": mean_absolute_error(y_te, pred_te),
        "RMSE": np.sqrt(mean_squared_error(y_te, pred_te)),
        "R2 train": r2_score(y_tr, pred_tr),
        "R2 test": r2_score(y_te, pred_te),
    }

results = [evaluate(model, X_train, y_train, X_test, y_test, "LinearRegression")]
pd.DataFrame(results).round(3)

### Коэффициенты модели

In [ ]:
coefs = pd.Series(model[-1].coef_, index=features).sort_values()
coefs.plot(kind="barh", figsize=(8, 4), title="Коэффициенты (на стандартизированных признаках)")
plt.axvline(0, color="black", lw=0.8); plt.show()
print("Intercept:", round(model[-1].intercept_, 3))
coefs.round(3)

### Предсказания vs реальные значения

In [ ]:
pred = model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, pred, alpha=0.2, s=8)
axes[0].plot([0, 5], [0, 5], "r--", label="идеальное предсказание")
axes[0].set_xlabel("Реальная цена"); axes[0].set_ylabel("Предсказанная цена")
axes[0].set_title("Предсказание vs реальность"); axes[0].legend()

sns.histplot(y_test - pred, bins=50, kde=True, ax=axes[1])
axes[1].set_title("Распределение остатков (y - ŷ)")
plt.tight_layout(); plt.show()

### Дополнительно: модель без «обрезанных» домов
Попробуем убрать дома с ценой 5.0 и экстремальные выбросы по `AveOccup`, `AveRooms` — и посмотрим, изменится ли качество.

In [ ]:
mask = (df["MedHouseVal"] < 5.0) & (df["AveOccup"] < 10) & (df["AveRooms"] < 20)
df_clean = df[mask]
print("Удалено строк:", len(df) - len(df_clean))

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    df_clean[features], df_clean["MedHouseVal"], test_size=0.2, random_state=42)

model_clean = make_pipeline(StandardScaler(), LinearRegression()).fit(Xc_train, yc_train)
results.append(evaluate(model_clean, Xc_train, yc_train, Xc_test, yc_test, "LinearRegression (без выбросов)"))
pd.DataFrame(results).round(3)

## Выводы
*(сверь цифры с таблицами выше после запуска)*

- Линейная регрессия на всех 8 признаках даёт **R² ≈ 0.58** на тесте, **MAE ≈ 0.53**, т.е. в среднем ошибается примерно на 53 000 $. R² на train и test почти одинаковые — переобучения нет, модель скорее **недообучена** (слишком простая).
- Самые важные коэффициенты: **`MedInc`** (чем выше доход, тем дороже дома), а также `Latitude` и `Longitude` с отрицательными знаками (чем севернее и восточнее — от побережья вглубь штата — тем дешевле). `AveRooms` и `AveBedrms` имеют противоположные знаки из-за мультиколлинеарности, поэтому их коэффициенты нельзя интерпретировать по отдельности.
- На графике «предсказание vs реальность» видно, что модель **занижает цену дорогих домов** (особенно обрезанных на 5.0) и иногда предсказывает отрицательные/завышенные цены для выбросов.
- Удаление обрезанных значений и выбросов немного улучшает метрики, но принципиально не меняет картину.
- **Почему качество ограничено:** зависимость цены от координат и многих признаков нелинейная, а линейная модель умеет описывать только прямые зависимости. Улучшить можно с помощью новых признаков (например, комнат на человека, расстояния до побережья/крупных городов), логарифмирования скошенных признаков или нелинейных моделей (деревья, случайный лес, бустинг).